# 🔍 Ética de Datos y Sesgo en Análisis Socioeconómico

**Módulo 01** | **Sesión 2** | **Duración estimada: 1h 30min**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-01-fundamentos/notebooks/01_04_etica_datos_sesgo.ipynb)

## 🎯 Objetivos de Aprendizaje

| # | Competencia | Descripción | Nivel |
|---|-------------|-------------|-------|
| 1 | Comprensión ética | Identificar los principios fundamentales de la ética de datos (FAIR, consentimiento, privacidad) | Conocimiento |
| 2 | Detección de sesgos | Reconocer y detectar sesgos comunes en datos socioeconómicos y universitarios | Aplicación |
| 3 | Anonimización | Aplicar técnicas básicas de anonimización para proteger la privacidad de los datos | Aplicación |
| 4 | Marco ético docente | Evaluar las implicaciones éticas del uso de datos en la investigación y docencia universitaria | Evaluación |

## 💡 ¿Por qué es importante para tu práctica docente?

Como docente de FACES-UC, trabajas constantemente con datos de estudiantes: calificaciones, tasas de aprobación, encuestas de satisfacción, información socioeconómica. Cada vez que analizas estos datos para tomar decisiones —ya sea ajustar tu metodología, diseñar una investigación, o elaborar un informe institucional— estás asumiendo una **responsabilidad ética**.

Un análisis sesgado puede llevar a conclusiones erróneas: creer que una carrera tiene mejor rendimiento cuando en realidad solo estás viendo a los estudiantes que no desertaron, o publicar datos que permitan identificar a un docente específico en una encuesta supuestamente anónima.

En esta sesión aprenderemos a:
- **Identificar** cuándo un análisis puede estar sesgado
- **Proteger** la privacidad de estudiantes y colegas
- **Aplicar** un marco ético riguroso en nuestra práctica como docentes-investigadores

---

## 📦 Configuración Inicial

Primero, importamos las librerías necesarias y cargamos los datos que usaremos durante toda la sesión.

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

# Configuración visual
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

# Paleta de colores institucional
COLORES = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']
sns.set_palette(COLORES)

print('✅ Librerías cargadas correctamente')

In [ ]:
# Cargar los tres datasets que usaremos
rendimiento = pd.read_csv('../../datasets/universidad/rendimiento_academico.csv')
matricula = pd.read_csv('../../datasets/universidad/matricula_faces.csv')
encuesta = pd.read_csv('../../datasets/universidad/encuesta_docente.csv')

print('Datasets cargados:')
print(f'  📊 Rendimiento académico: {rendimiento.shape[0]:,} registros, {rendimiento.shape[1]} columnas')
print(f'  📊 Matrícula FACES:        {matricula.shape[0]:,} registros, {matricula.shape[1]} columnas')
print(f'  📊 Encuesta docente:      {encuesta.shape[0]:,} registros, {encuesta.shape[1]} columnas')

---

## 📖 Sección 1: ¿Qué es la ética de datos?

La **ética de datos** es el conjunto de principios morales que guían cómo recopilamos, almacenamos, analizamos y compartimos datos. En el contexto universitario, esto es especialmente relevante porque trabajamos con información sensible de personas reales: estudiantes, docentes y personal administrativo.

### Los Principios FAIR

Los principios **FAIR** (Wilkinson et al., 2016) son un estándar internacional para la gestión de datos científicos:

| Principio | Significado | Ejemplo en FACES |
|-----------|-------------|-------------------|
| **F**indable (Encontrable) | Los datos deben tener identificadores únicos y metadatos ricos | Cada estudiante tiene un `estudiante_id` único |
| **A**ccessible (Accesible) | Los datos deben ser recuperables mediante protocolos estándar | Los datos de matrícula están en formato CSV estándar |
| **I**nteroperable (Interoperable) | Los datos usan vocabularios y formatos compartidos | Las carreras usan nombres consistentes en todos los archivos |
| **R**eusable (Reutilizable) | Los datos tienen licencias claras y procedencia documentada | Se documenta cómo y cuándo se recopilaron los datos |

### Consentimiento Informado

Antes de recopilar datos de cualquier persona, debemos asegurar que:

1. **Saben qué datos se recopilan** — No recoger información oculta
2. **Entienden para qué se usarán** — Propósito claro y específico
3. **Pueden negarse sin consecuencias** — Participación voluntaria
4. **Pueden solicitar la eliminación** — Derecho al olvido

### Privacidad

La privacidad no es solo "no publicar nombres". Implica que **no sea posible re-identificar** a una persona a partir de los datos publicados. Como veremos más adelante, esto requiere técnicas específicas de anonimización.

Veamos un ejemplo concreto. ¿Qué información tienen nuestros datasets? ¿Cuáles columnas podrían ser sensibles?

In [ ]:
# Explorar qué datos tenemos y cuáles podrían ser sensibles
print('=' * 65)
print('COLUMNAS DEL DATASET DE RENDIMIENTO ACADÉMICO')
print('=' * 65)

for columna in rendimiento.columns:
    ejemplo = rendimiento[columna].iloc[0]
    tipo = rendimiento[columna].dtype
    # Clasificar sensibilidad
    if columna in ['estudiante_id']:
        sensibilidad = '🔴 IDENTIFICADOR DIRECTO'
    elif columna in ['edad', 'genero']:
        sensibilidad = '🟡 CUASI-IDENTIFICADOR'
    elif columna in ['promedio', 'materias_reprobadas']:
        sensibilidad = '🟠 DATO SENSIBLE'
    else:
        sensibilidad = '🟢 DATO GENERAL'
    
    print(f'  {columna:25s} | Tipo: {str(tipo):8s} | Ej: {str(ejemplo):20s} | {sensibilidad}')

print()
print('💡 Nota: Los cuasi-identificadores (edad + género + carrera) pueden')
print('   combinarse para re-identificar a una persona específica.')

---

## ⚠️ Sección 2: Sesgos en datos socioeconómicos

Un **sesgo** en datos es una distorsión sistemática que hace que los datos no representen fielmente la realidad. En el ámbito socioeconómico y universitario, los sesgos son especialmente peligrosos porque pueden afectar políticas académicas, distribución de recursos y evaluaciones de programas.

### Tipos de sesgo más comunes

| Tipo de sesgo | Descripción | Ejemplo universitario |
|---------------|-------------|----------------------|
| **Sesgo de selección** | La muestra no es representativa de la población | Encuestar solo a estudiantes presenciales, ignorando a los que no asisten |
| **Sesgo de supervivencia** | Solo vemos a los que "sobrevivieron" al proceso | Calcular el promedio solo con estudiantes activos, ignorando a los que desertaron |
| **Sesgo de medición** | El instrumento de medición introduce distorsiones | Escalas de satisfacción donde "5" significa cosas diferentes para cada persona |
| **Sesgo de confirmación** | Buscamos datos que confirmen lo que ya creemos | Seleccionar solo las estadísticas que apoyan nuestra hipótesis |

### Ejemplo práctico: Sesgo de supervivencia

Imaginemos que queremos evaluar el rendimiento académico en FACES. Si solo analizamos a los estudiantes que siguen activos, estamos ignorando a todos los que desertaron — y probablemente desertaron precisamente porque tenían bajo rendimiento. Esto infla artificialmente el promedio.

In [ ]:
# Demostrar el sesgo de supervivencia con datos de rendimiento
print('ANÁLISIS DE RENDIMIENTO ACADÉMICO')
print('=' * 50)

# Estadísticas generales
promedio_general = rendimiento['promedio'].mean()
mediana_general = rendimiento['promedio'].median()
print(f'\nPromedio general (todos los estudiantes): {promedio_general:.2f}')
print(f'Mediana general:                          {mediana_general:.2f}')
print(f'Total de estudiantes:                     {len(rendimiento):,}')

# Ahora simulemos el sesgo de supervivencia:
# Los estudiantes con promedio muy bajo probablemente desertarían
# Filtremos solo a los que "sobrevivieron" (promedio >= 10, que es lo mínimo para aprobar)
sobrevivientes = rendimiento[rendimiento['promedio'] >= 10]

promedio_sobrevivientes = sobrevivientes['promedio'].mean()
print(f'\n--- Si solo analizamos estudiantes con promedio >= 10 ---')
print(f'Promedio (solo "sobrevivientes"):          {promedio_sobrevivientes:.2f}')
print(f'Estudiantes incluidos:                    {len(sobrevivientes):,} de {len(rendimiento):,}')
print(f'Estudiantes EXCLUIDOS:                    {len(rendimiento) - len(sobrevivientes):,}')
print(f'\n⚠️  Diferencia por sesgo de supervivencia:   +{promedio_sobrevivientes - promedio_general:.2f} puntos')
print(f'   Esto representa un {((promedio_sobrevivientes - promedio_general) / promedio_general * 100):.1f}% de inflación en el promedio.')

In [ ]:
# Visualizar la diferencia entre la distribución completa y la sesgada
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución completa
axes[0].hist(rendimiento['promedio'], bins=30, color=COLORES[0], alpha=0.7, edgecolor='white')
axes[0].axvline(promedio_general, color=COLORES[3], linewidth=2, linestyle='--', 
                label=f'Promedio: {promedio_general:.1f}')
axes[0].set_title('Todos los estudiantes\n(Visión completa)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Promedio de calificaciones')
axes[0].set_ylabel('Número de estudiantes')
axes[0].legend(fontsize=11)

# Distribución sesgada (solo "sobrevivientes")
axes[1].hist(sobrevivientes['promedio'], bins=25, color=COLORES[1], alpha=0.7, edgecolor='white')
axes[1].axvline(promedio_sobrevivientes, color=COLORES[3], linewidth=2, linestyle='--',
                label=f'Promedio: {promedio_sobrevivientes:.1f}')
axes[1].set_title('Solo estudiantes activos (promedio ≥ 10)\n(⚠️ Visión sesgada)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Promedio de calificaciones')
axes[1].set_ylabel('Número de estudiantes')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.suptitle('Sesgo de Supervivencia en Rendimiento Académico', 
             fontsize=15, fontweight='bold', y=1.03)
plt.show()

print('\n💡 Observa cómo al excluir a los estudiantes con bajo rendimiento,') 
print('   la distribución se desplaza hacia la derecha y el promedio sube.')
print('   Esto podría dar una imagen demasiado optimista del rendimiento real.')

In [ ]:
# Sesgo de selección: ¿Qué pasa si solo encuestamos a quienes asisten?
print('SESGO DE SELECCIÓN: Asistencia y rendimiento')
print('=' * 50)

# Dividir por asistencia (los que asisten regularmente vs los que no)
asistencia_alta = rendimiento[rendimiento['asistencia_pct'] >= 80]
asistencia_baja = rendimiento[rendimiento['asistencia_pct'] < 80]

print(f'\nEstudiantes con asistencia >= 80%: {len(asistencia_alta):,} ({len(asistencia_alta)/len(rendimiento)*100:.1f}%)')
print(f'  Promedio de este grupo: {asistencia_alta["promedio"].mean():.2f}')
print(f'\nEstudiantes con asistencia < 80%:  {len(asistencia_baja):,} ({len(asistencia_baja)/len(rendimiento)*100:.1f}%)')
print(f'  Promedio de este grupo: {asistencia_baja["promedio"].mean():.2f}')

print(f'\n⚠️  Si hacemos una encuesta en el aula, solo capturamos al grupo')
print(f'   de alta asistencia. Su promedio ({asistencia_alta["promedio"].mean():.2f}) NO representa')
print(f'   al total de la sección (promedio real: {rendimiento["promedio"].mean():.2f}).')

---

## 🏫 Sección 3: Caso práctico — Sesgo en datos universitarios

Ahora trabajaremos con los datos reales de matrícula de FACES. Este dataset contiene información de todos los estudiantes inscritos entre 2015 y 2025, incluyendo su **estatus** actual: Activo, Egresado, o Retirado.

La pregunta clave: **¿Qué pasa con nuestro análisis si ignoramos a los estudiantes retirados?**

In [ ]:
# Explorar el dataset de matrícula
print('DATASET DE MATRÍCULA FACES')
print('=' * 50)
print(f'\nPeríodos: {matricula["periodo"].nunique()} (desde {matricula["periodo"].min()} hasta {matricula["periodo"].max()})')
print(f'Total de registros: {len(matricula):,}')
print(f'\nDistribución por estatus:')
estatus_counts = matricula['estatus'].value_counts()
for estatus, count in estatus_counts.items():
    pct = count / len(matricula) * 100
    print(f'  {estatus:12s}: {count:,} ({pct:.1f}%)')

print(f'\nDistribución por carrera:')
for carrera in sorted(matricula['carrera'].unique()):
    count = len(matricula[matricula['carrera'] == carrera])
    print(f'  {carrera:25s}: {count:,}')

In [ ]:
# Comparar composición por género: CON y SIN retirados
print('COMPOSICIÓN POR GÉNERO: ¿Cambia si excluimos retirados?')
print('=' * 55)

# Análisis completo (todos los estatus)
genero_total = matricula['genero'].value_counts(normalize=True) * 100
print(f'\nTodos los estudiantes:')
for g, pct in genero_total.items():
    etiqueta = 'Femenino' if g == 'F' else 'Masculino'
    print(f'  {etiqueta}: {pct:.1f}%')

# Análisis sin retirados (visión sesgada)
sin_retirados = matricula[matricula['estatus'] != 'Retirado']
genero_sin_ret = sin_retirados['genero'].value_counts(normalize=True) * 100
print(f'\nSin retirados:')
for g, pct in genero_sin_ret.items():
    etiqueta = 'Femenino' if g == 'F' else 'Masculino'
    print(f'  {etiqueta}: {pct:.1f}%')

# Análisis solo de retirados
retirados = matricula[matricula['estatus'] == 'Retirado']
genero_retirados = retirados['genero'].value_counts(normalize=True) * 100
print(f'\nSolo retirados:')
for g, pct in genero_retirados.items():
    etiqueta = 'Femenino' if g == 'F' else 'Masculino'
    print(f'  {etiqueta}: {pct:.1f}%')

print(f'\n💡 Observa si la composición de género entre los retirados difiere')
print(f'   del total. Si es así, excluirlos introduce un sesgo de género.')

In [ ]:
# Visualizar el impacto de excluir retirados por carrera
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

carreras = sorted(matricula['carrera'].unique())

# Gráfico 1: Tasa de retiro por carrera
tasa_retiro = []
for carrera in carreras:
    datos_carrera = matricula[matricula['carrera'] == carrera]
    tasa = len(datos_carrera[datos_carrera['estatus'] == 'Retirado']) / len(datos_carrera) * 100
    tasa_retiro.append(tasa)

barras = axes[0].bar(carreras, tasa_retiro, color=COLORES[:4], edgecolor='white', linewidth=1.5)
axes[0].set_title('Tasa de retiro por carrera', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Porcentaje de retirados (%)')
axes[0].tick_params(axis='x', rotation=30)
for barra, tasa in zip(barras, tasa_retiro):
    axes[0].text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 0.3,
                f'{tasa:.1f}%', ha='center', fontweight='bold', fontsize=11)

# Gráfico 2: Comparar distribución de edad CON y SIN retirados
axes[1].hist(matricula['edad'], bins=20, alpha=0.5, label='Todos', color=COLORES[0], edgecolor='white')
axes[1].hist(sin_retirados['edad'], bins=20, alpha=0.5, label='Sin retirados', color=COLORES[1], edgecolor='white')
axes[1].axvline(matricula['edad'].mean(), color=COLORES[0], linestyle='--', linewidth=2,
               label=f'Media todos: {matricula["edad"].mean():.1f}')
axes[1].axvline(sin_retirados['edad'].mean(), color=COLORES[1], linestyle='--', linewidth=2,
               label=f'Media sin ret.: {sin_retirados["edad"].mean():.1f}')
axes[1].set_title('Distribución de edad: impacto de excluir retirados', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Edad')
axes[1].set_ylabel('Frecuencia')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print('\n⚠️  Excluir retirados puede cambiar las conclusiones sobre:') 
print('   - La composición demográfica de cada carrera')
print('   - La distribución etaria de los estudiantes')
print('   - Las tasas de éxito aparentes')

In [ ]:
# Análisis temporal: ¿cómo varía la tasa de retiro por período?
periodos_orden = sorted(matricula['periodo'].unique())

tasas_por_periodo = []
for periodo in periodos_orden:
    datos_periodo = matricula[matricula['periodo'] == periodo]
    total = len(datos_periodo)
    retirados_p = len(datos_periodo[datos_periodo['estatus'] == 'Retirado'])
    tasas_por_periodo.append(retirados_p / total * 100)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(periodos_orden, tasas_por_periodo, marker='o', linewidth=2, color=COLORES[3], markersize=6)
ax.fill_between(periodos_orden, tasas_por_periodo, alpha=0.15, color=COLORES[3])
ax.set_title('Evolución de la tasa de retiro por período académico', fontsize=14, fontweight='bold')
ax.set_xlabel('Período')
ax.set_ylabel('Tasa de retiro (%)')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n💡 Si la tasa de retiro cambia significativamente entre períodos,')
print('   el sesgo de excluir retirados será diferente según la cohorte analizada.')

---

## 🔒 Sección 4: Privacidad y anonimización

### Información de Identificación Personal (PII)

La **PII** (Personally Identifiable Information) es cualquier dato que puede usarse para identificar a una persona específica, ya sea directamente o en combinación con otros datos:

- **Identificadores directos**: Nombre, cédula, correo electrónico, número de empleado
- **Cuasi-identificadores**: Edad exacta, género, departamento, combinaciones de atributos

### K-anonimidad

El concepto de **k-anonimidad** establece que cada registro en un dataset debe ser indistinguible de al menos *k-1* otros registros en sus cuasi-identificadores. En términos simples:

> Si publico datos donde digo que hay un docente de 58 años, hombre, del departamento de Estadística... ¿cuántos docentes en FACES cumplen esas condiciones? Si solo hay uno, ¡lo he identificado!

Con **k=5**, necesitamos que al menos 5 personas compartan cada combinación de cuasi-identificadores.

### Técnicas de anonimización

| Técnica | Descripción | Ejemplo |
|---------|-------------|----------|
| **Supresión** | Eliminar columnas identificadoras | Quitar `docente_id`, nombre, cédula |
| **Generalización** | Reemplazar valores exactos por rangos | Edad 58 → Rango "55-64" |
| **Perturbación** | Añadir ruido aleatorio a valores numéricos | Edad 58 → 56 o 60 (aleatoriamente) |
| **Agregación** | Publicar solo resúmenes, no datos individuales | No publicar registros, solo promedios por grupo |

In [ ]:
# Veamos el dataset de encuesta docente - ¿qué tan identificable es?
print('DATASET DE ENCUESTA DOCENTE')
print('=' * 50)
print()
print(encuesta.head(10).to_string(index=False))
print(f'\nTotal de docentes: {len(encuesta)}')

In [ ]:
# Evaluar riesgo de re-identificación
# ¿Cuántas combinaciones únicas de cuasi-identificadores existen?
cuasi_ids = ['departamento', 'genero', 'edad', 'dedicacion']

print('ANÁLISIS DE K-ANONIMIDAD')
print('=' * 50)
print(f'\nCuasi-identificadores analizados: {cuasi_ids}')

# Contar cuántos docentes comparten cada combinación
grupos = encuesta.groupby(cuasi_ids).size().reset_index(name='k')

print(f'\nCombinaciones únicas: {len(grupos)}')
print(f'\nDistribución de k (tamaño de grupo):')
for k_val in sorted(grupos['k'].unique()):
    count = len(grupos[grupos['k'] == k_val])
    if k_val <= 3:  # Resaltar los riesgosos
        marcador = '🔴' if k_val == 1 else '🟡'
    else:
        marcador = '🟢'
    print(f'  {marcador} k={k_val}: {count} combinaciones')

# Cuántos son únicos (k=1)?
unicos = len(grupos[grupos['k'] == 1])
docentes_unicos = grupos[grupos['k'] == 1].merge(encuesta, on=cuasi_ids).shape[0]
print(f'\n⚠️  {docentes_unicos} docentes son únicamente identificables con estos cuasi-identificadores.')
print(f'   Esto significa que si se publican estos datos tal cual, esas personas')
print(f'   podrían ser reconocidas por cualquiera que conozca su departamento, género y edad.')

In [ ]:
# Aplicar técnicas de anonimización paso a paso
print('PROCESO DE ANONIMIZACIÓN')
print('=' * 50)

# Paso 1: Supresión - eliminar el identificador directo
encuesta_anon = encuesta.drop(columns=['docente_id']).copy()
print('\n✅ Paso 1 - SUPRESIÓN: Eliminada la columna docente_id')
print(f'   Columnas restantes: {list(encuesta_anon.columns)}')

# Paso 2: Generalización - convertir edad exacta en rangos
bins_edad = [0, 35, 45, 55, 100]
etiquetas_edad = ['Menor de 35', '35-44', '45-54', '55 o más']
encuesta_anon['edad'] = pd.cut(encuesta_anon['edad'], bins=bins_edad, labels=etiquetas_edad)
print(f'\n✅ Paso 2 - GENERALIZACIÓN: Edad convertida a rangos')
print(f'   Rangos: {etiquetas_edad}')
print(f'   Distribución:')
for rango, count in encuesta_anon['edad'].value_counts().sort_index().items():
    print(f'     {rango}: {count} docentes')

# Paso 3: Generalización - convertir antigüedad en rangos
bins_ant = [0, 5, 10, 20, 100]
etiquetas_ant = ['0-5 años', '6-10 años', '11-20 años', 'Más de 20']
encuesta_anon['antiguedad'] = pd.cut(encuesta_anon['antiguedad'], bins=bins_ant, labels=etiquetas_ant)
print(f'\n✅ Paso 3 - GENERALIZACIÓN: Antigüedad convertida a rangos')
print(f'   Rangos: {etiquetas_ant}')

In [ ]:
# Verificar la mejora en k-anonimidad después de la anonimización
cuasi_ids_anon = ['departamento', 'genero', 'edad', 'dedicacion']

grupos_anon = encuesta_anon.groupby(cuasi_ids_anon).size().reset_index(name='k')

print('K-ANONIMIDAD DESPUÉS DE ANONIMIZACIÓN')
print('=' * 50)
print(f'\nAntes: {len(grupos)} combinaciones únicas')
print(f'Después: {len(grupos_anon)} combinaciones únicas')
print(f'\nDistribución de k después de anonimizar:')
for k_val in sorted(grupos_anon['k'].unique()):
    count = len(grupos_anon[grupos_anon['k'] == k_val])
    marcador = '🔴' if k_val == 1 else ('🟡' if k_val < 5 else '🟢')
    print(f'  {marcador} k={k_val}: {count} combinaciones')

unicos_despues = len(grupos_anon[grupos_anon['k'] == 1])
print(f'\nMejora: de {unicos} a {unicos_despues} combinaciones con k=1')
print(f'\nMuestra del dataset anonimizado:')
print(encuesta_anon.head(8).to_string(index=False))

In [ ]:
# Visualizar la mejora en k-anonimidad
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Antes de anonimizar
k_antes = grupos['k'].value_counts().sort_index()
axes[0].bar(k_antes.index.astype(str), k_antes.values, color=COLORES[3], edgecolor='white')
axes[0].set_title('Antes de anonimizar', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Valor de k')
axes[0].set_ylabel('Número de combinaciones')
axes[0].axhline(y=0, color='black', linewidth=0.5)

# Después de anonimizar
k_despues = grupos_anon['k'].value_counts().sort_index()
axes[1].bar(k_despues.index.astype(str), k_despues.values, color=COLORES[0], edgecolor='white')
axes[1].set_title('Después de anonimizar', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Valor de k')
axes[1].set_ylabel('Número de combinaciones')
axes[1].axhline(y=0, color='black', linewidth=0.5)

plt.suptitle('Mejora en k-anonimidad tras generalización de edad y antigüedad',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n💡 Después de la generalización, los valores de k son más grandes,')
print('   lo que significa mayor protección contra la re-identificación.')

---

## 📜 Sección 5: Marco ético para el docente-investigador

### Principios LOPED

Los principios **LOPED** proporcionan un marco práctico para la ética de datos en investigación:

| Principio | Descripción | Aplicación en FACES |
|-----------|-------------|---------------------|
| **L**egalidad | Cumplir con leyes y regulaciones aplicables | Ley Orgánica de Protección de Datos Personales (LOPD) en Venezuela |
| **O**bjetividad | Evitar sesgos intencionales en el análisis | No manipular datos para obtener resultados deseados |
| **P**roporcionalidad | Recopilar solo los datos necesarios | No pedir cédula si no es estrictamente necesario |
| **E**xactitud | Mantener datos precisos y actualizados | Verificar periódicamente que las bases de datos estén vigentes |
| **D**eber de confidencialidad | Proteger los datos de acceso no autorizado | Cifrar archivos, limitar acceso, anonimizar para publicaciones |

### Comités de Ética e Investigación

En el contexto universitario venezolano, toda investigación que involucre datos de personas debe:

1. **Obtener aprobación** del Comité de Ética o Consejo de Investigación de la facultad
2. **Documentar** el consentimiento informado de los participantes
3. **Garantizar** que los datos se almacenen de forma segura
4. **Especificar** el período de retención de los datos
5. **Describir** las medidas de anonimización aplicadas

### Lista de verificación ética

Antes de realizar cualquier análisis con datos reales, revisa esta lista:

- [ ] ¿Los datos fueron recopilados con consentimiento?
- [ ] ¿Estoy usando solo los datos necesarios para mi análisis?
- [ ] ¿He eliminado los identificadores directos?
- [ ] ¿He verificado que no se pueda re-identificar a nadie?
- [ ] ¿Mi muestra es representativa o tiene sesgos conocidos?
- [ ] ¿He documentado las limitaciones de mis datos?
- [ ] ¿Mis conclusiones son proporcionales a la evidencia?
- [ ] ¿He considerado el impacto de mis hallazgos en las personas involucradas?

In [ ]:
# Ejemplo: creación de un reporte ético a partir de datos de encuesta
# Demostración de cómo publicar resultados sin comprometer la privacidad

print('EJEMPLO: Reporte ético de satisfacción docente')
print('=' * 55)
print()
print('✅ CORRECTO: Publicar resultados agregados')
print('-' * 55)

# Resultados agregados por departamento (seguros de publicar)
resumen = encuesta.groupby('departamento').agg(
    n_docentes=('docente_id', 'count'),
    satisfaccion_media=('satisfaccion_general', 'mean'),
    publicaciones_media=('publicaciones', 'mean')
).round(2)

print(resumen.to_string())

print(f'\n\n❌ INCORRECTO: Publicar datos individuales sin anonimizar')
print('-' * 55)
print('  Ejemplo problemático:')
ejemplo_malo = encuesta[encuesta['departamento'] == 'Estadistica'][['departamento', 'genero', 'edad', 'satisfaccion_general', 'publicaciones']].head(3)
print(f'  {ejemplo_malo.to_string(index=False)}')
print(f'\n  ⚠️ Con solo 3 variables (departamento, género, edad) se podría')
print(f'     identificar a un docente específico en un departamento pequeño.')

---

## ✏️ Ejercicios

### Ejercicio 1: Identificar sesgos potenciales en el dataset de rendimiento académico

Analiza el dataset `rendimiento_academico.csv` y responde:

1. **Sesgo de selección por trabajo**: ¿Los estudiantes que trabajan tienen un rendimiento diferente? Si solo analizáramos a los que NO trabajan, ¿cómo cambiarían las conclusiones?
2. **Sesgo por género**: ¿La proporción de becados es igual entre géneros? Si no, ¿qué sesgo podría esto introducir?
3. Escribe código para detectar y cuantificar al menos UN sesgo en los datos.

**Pista**: Compara las distribuciones de promedio entre diferentes subgrupos.

In [ ]:
# EJERCICIO 1: Identificar sesgos en rendimiento_academico.csv
# -----------------------------------------------------------

# 1a. Comparar rendimiento entre estudiantes que trabajan y los que no
# Tu código aquí


# 1b. Verificar si la proporción de becados varía por género
# Tu código aquí


# 1c. Cuantificar el sesgo encontrado: ¿cuánto cambia el promedio
#      si excluimos un subgrupo específico?
# Tu código aquí


### Ejercicio 2: Anonimizar el dataset de encuesta docente

Aplica un proceso de anonimización al dataset `encuesta_docente.csv` para que sea seguro publicarlo. Debes:

1. **Eliminar** el identificador directo (`docente_id`)
2. **Generalizar** la edad en rangos de 10 años (28-37, 38-47, 48-57, 58+)
3. **Generalizar** la antigüedad en rangos (0-5, 6-15, 16+)
4. **Verificar** la k-anonimidad resultante usando los cuasi-identificadores: departamento, género, rango de edad, dedicación
5. **Reportar**: ¿Cuál es el k mínimo alcanzado? ¿Es suficiente para publicar?

**Objetivo**: Lograr un k mínimo de al menos 3.

In [ ]:
# EJERCICIO 2: Anonimizar encuesta_docente.csv
# -----------------------------------------------------------

# Cargar datos frescos
encuesta_ej = pd.read_csv('../../datasets/universidad/encuesta_docente.csv')

# Paso 1: Eliminar el identificador directo
# Tu código aquí


# Paso 2: Generalizar la edad en rangos de 10 años
# Pista: usa pd.cut() con bins=[27, 37, 47, 57, 100]
# Tu código aquí


# Paso 3: Generalizar la antigüedad en rangos
# Tu código aquí


# Paso 4: Verificar k-anonimidad
# Pista: agrupa por los cuasi-identificadores y cuenta el tamaño de cada grupo
# Tu código aquí


# Paso 5: ¿Cuál es el k mínimo? ¿Es seguro publicar?
# Tu código aquí


---

## 📋 Resumen

En esta sesión aprendimos conceptos fundamentales para el manejo ético de datos en el contexto universitario:

| Concepto | Punto clave |
|----------|-------------|
| **Principios FAIR** | Los datos deben ser encontrables, accesibles, interoperables y reutilizables |
| **Sesgo de supervivencia** | Analizar solo a quienes "sobreviven" (no desertan, no se retiran) infla los resultados |
| **Sesgo de selección** | La muestra debe representar a toda la población, no solo a los más accesibles |
| **PII y cuasi-identificadores** | La combinación de datos aparentemente inocuos puede identificar a una persona |
| **K-anonimidad** | Cada persona debe ser indistinguible de al menos k-1 otras en los cuasi-identificadores |
| **Anonimización** | Supresión, generalización y perturbación son técnicas complementarias |
| **Marco ético LOPED** | Legalidad, Objetividad, Proporcionalidad, Exactitud y Deber de confidencialidad |

### Para llevar a tu práctica

- **Antes de analizar**: Verifica que tus datos no tienen sesgos ocultos (de supervivencia, de selección, de medición)
- **Antes de publicar**: Anonimiza los datos y verifica la k-anonimidad
- **Siempre**: Documenta las limitaciones de tus datos y las decisiones éticas que tomaste

## 📚 Referencias

1. Wilkinson, M. D., et al. (2016). *The FAIR Guiding Principles for scientific data management and stewardship*. Scientific Data, 3, 160018. https://doi.org/10.1038/sdata.2016.18

2. Sweeney, L. (2002). *k-Anonymity: A model for protecting privacy*. International Journal of Uncertainty, Fuzziness and Knowledge-Based Systems, 10(5), 557-570.

3. Floridi, L., & Taddeo, M. (2016). *What is data ethics?* Philosophical Transactions of the Royal Society A, 374(2083), 20160360.

4. Asamblea Nacional de Venezuela (2001). *Constitución de la República Bolivariana de Venezuela*. Artículos 28, 60 (protección de datos personales y privacidad).

5. UNESCO (2021). *Recomendación sobre la Ética de la Inteligencia Artificial*. https://www.unesco.org/es/artificial-intelligence/recommendation-ethics

6. Kitchin, R. (2014). *The Data Revolution: Big Data, Open Data, Data Infrastructures and Their Consequences*. SAGE Publications.